# Discrete Leg-Step Brain Playground

Train a tiny linear policy on the 81-action discrete-leg harness. At each
brain step (default 0.2s), the policy picks one of `3^4 = 81` options;
each leg either stays put, steps +45 deg, or steps -45 deg. Reward per
step is the reduction in distance from the body to the goal. Episodes end
early on tipover or once the body comes within `goal_reached_radius_m`
of the goal (small but non-zero, to absorb numerical offsets).

Self-contained: tiny JAX linear policy + Gaussian ES, no spec/registry juggling.

Rollouts use the static MJCF/STL robot assets in `assets/mujoco/`.

In [ ]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import math
import sys
import time

import numpy as np
import jax
import jax.numpy as jnp
from jax.flatten_util import ravel_pytree

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "train_headless.py").exists() and (REPO_ROOT.parent / "train_headless.py").exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from brains.harnesses import DiscreteLegHarness, NUM_ACTIONS

print("jax devices:", jax.devices())
print("NUM_ACTIONS:", NUM_ACTIONS)

## Tunable hyperparameters

Tweak these in-place and re-run the cells below.

In [ ]:
# --- harness knobs ---
SEED = 11
BRAIN_DT_S = 0.20                # seconds per discrete decision (45 deg fits in this window)
GOAL_REACHED_RADIUS_M = 0.30     # "close enough" early-stop radius (small but non-zero)
GOAL_DISTANCE_M = 3.0            # initial distance to goal along +x
GOAL_HEIGHT_M = 0.16             # only affects body[:2] vs goal[:2] in obs; kept for completeness
TIP_KILL_DEPTH = 0.9             # 0..1 normalized side-tip depth that counts as 'tipping'
TIP_KILL_STEPS = 2               # consecutive tipping steps that end the episode
POSITIONAL_ENCODING_GAIN = 0.35  # 0 disables, otherwise blends in sin/cos features
GOAL_REACHED_BONUS = 5.0         # terminal reward for reaching the goal
TIPPED_PENALTY = 0.5             # subtracted on tipover (0 = no extra penalty)

# --- training knobs ---
MAX_EPISODE_STEPS = 200           # 80 * 0.2s = 16s per episode
POPULATION_SIZE = 100             # must be even (antithetic pairs)
SIGMA = 0.10                     # Gaussian noise scale on params
LEARNING_RATE = 0.05             # step size for ES update
GENERATIONS = 300                 # number of ES outer iterations
EVAL_EPISODES_PER_VARIANT = 1    # rollouts per population member (avg reward)
PRINT_EVERY = 1                  # print after every N generations

assert POPULATION_SIZE % 2 == 0, "POPULATION_SIZE must be even (antithetic ES sampling)."

{
    "brain_dt_s": BRAIN_DT_S,
    "max_episode_seconds": MAX_EPISODE_STEPS * BRAIN_DT_S,
    "num_actions": NUM_ACTIONS,
}

In [ ]:
harness = DiscreteLegHarness(
    brain_dt_s=BRAIN_DT_S,
    goal_reached_radius_m=GOAL_REACHED_RADIUS_M,
    tip_kill_depth=TIP_KILL_DEPTH,
    tip_kill_steps=TIP_KILL_STEPS,
    positional_encoding_gain=POSITIONAL_ENCODING_GAIN,
    goal_reached_bonus=GOAL_REACHED_BONUS,
    tipped_penalty=TIPPED_PENALTY,
)

ENCODED_DIM = harness.encoded_obs_dim
GOAL_XYZ = np.asarray([GOAL_DISTANCE_M, 0.0, GOAL_HEIGHT_M], dtype=np.float32)
SPAWN_XY = np.zeros((2,), dtype=np.float32)

{
    "base_obs_dim": harness.BASE_OBS_DIM,
    "encoded_obs_dim": ENCODED_DIM,
    "linear_policy_param_count": ENCODED_DIM * NUM_ACTIONS + NUM_ACTIONS,
}

In [ ]:
def init_params(key: jax.Array, in_dim: int, out_dim: int = NUM_ACTIONS):
    key_w, key_b = jax.random.split(key)
    scale = jnp.float32(1.0 / math.sqrt(max(in_dim, 1)))
    return {
        "W": jax.random.normal(key_w, (out_dim, in_dim), dtype=jnp.float32) * scale,
        "b": jax.random.uniform(key_b, (out_dim,), dtype=jnp.float32, minval=-0.05, maxval=0.05),
    }

@jax.jit
def policy_logits(params, encoded_obs):
    return params["W"] @ encoded_obs + params["b"]

def make_chooser(params):
    def choose(obs_np, _step_index):
        encoded = harness.encode_obs(obs_np)
        logits = policy_logits(params, encoded)
        return int(jnp.argmax(logits))
    return choose

def evaluate(params, *, max_steps: int, episodes: int = 1, record_frames: bool = False):
    chooser = make_chooser(params)
    rewards: list[float] = []
    last_result = None
    for _ in range(int(episodes)):
        last_result = harness.rollout(
            chooser,
            max_steps=max_steps,
            spawn_xy=SPAWN_XY,
            goal_xyz=GOAL_XYZ,
            record_frames=record_frames,
        )
        rewards.append(float(last_result.total_reward))
    return float(np.mean(rewards)), last_result

key = jax.random.PRNGKey(SEED)
key, subkey = jax.random.split(key)
params = init_params(subkey, ENCODED_DIM)
warmup_reward, warmup_result = evaluate(params, max_steps=MAX_EPISODE_STEPS, episodes=1)
print(
    f"warmup random-init: reward={warmup_reward:.3f} steps={warmup_result.steps} "
    f"reached={warmup_result.reached_goal} tipped={warmup_result.tipped} "
    f"final_dist={warmup_result.final_distance_m:.3f}"
)

In [ ]:
def es_generation(params, key, *, sigma: float, learning_rate: float, population: int, max_steps: int, episodes_per_variant: int):
    flat, unravel = ravel_pytree(params)
    half = population // 2
    key, noise_key = jax.random.split(key)
    noise_half = jax.random.normal(noise_key, (half, flat.shape[0]), dtype=jnp.float32)
    noise = jnp.concatenate([noise_half, -noise_half], axis=0)

    rewards = np.zeros((population,), dtype=np.float32)
    n_reached = 0
    n_tipped = 0
    finish_steps: list[int] = []

    for member_index in range(population):
        candidate = unravel(flat + jnp.float32(sigma) * noise[member_index])
        member_reward = 0.0
        for _ in range(int(episodes_per_variant)):
            mean_reward, result = evaluate(candidate, max_steps=max_steps, episodes=1)
            member_reward += mean_reward
            if result.reached_goal:
                n_reached += 1
                finish_steps.append(int(result.steps))
            if result.tipped:
                n_tipped += 1
        rewards[member_index] = member_reward / float(episodes_per_variant)

    rewards_jax = jnp.asarray(rewards, dtype=jnp.float32)
    centered = rewards_jax - jnp.mean(rewards_jax)
    std = jnp.std(rewards_jax) + jnp.float32(1e-8)
    standardized = centered / std
    grad = (noise.T @ standardized) / jnp.float32(population * sigma)
    new_flat = flat + jnp.float32(learning_rate) * grad
    new_params = unravel(new_flat)

    rollouts = population * episodes_per_variant
    stats = {
        "mean_reward": float(jnp.mean(rewards_jax)),
        "max_reward": float(jnp.max(rewards_jax)),
        "min_reward": float(jnp.min(rewards_jax)),
        "frac_reached": float(n_reached) / float(rollouts),
        "frac_tipped": float(n_tipped) / float(rollouts),
        "frac_stopped_early": float(n_reached + n_tipped) / float(rollouts),
        "avg_finish_time_s": (float(np.mean(finish_steps)) * BRAIN_DT_S) if finish_steps else float("nan"),
        "finishers": int(n_reached),
    }
    return new_params, key, stats

history: list[dict] = []
train_start = time.perf_counter()

for generation_index in range(int(GENERATIONS)):
    gen_start = time.perf_counter()
    params, key, stats = es_generation(
        params,
        key,
        sigma=SIGMA,
        learning_rate=LEARNING_RATE,
        population=POPULATION_SIZE,
        max_steps=MAX_EPISODE_STEPS,
        episodes_per_variant=EVAL_EPISODES_PER_VARIANT,
    )
    stats["generation"] = generation_index + 1
    stats["elapsed_s"] = time.perf_counter() - gen_start
    history.append(stats)

    if (generation_index + 1) % PRINT_EVERY == 0:
        finish_str = (
            f"avg_finish_time={stats['avg_finish_time_s']:.2f}s"
            if stats["finishers"] > 0
            else "avg_finish_time=n/a"
        )
        print(
            f"gen {stats['generation']:>3d}/{GENERATIONS} | "
            f"reward mean={stats['mean_reward']:+.3f} max={stats['max_reward']:+.3f} | "
            f"reached={stats['frac_reached']*100:5.1f}% "
            f"tipped={stats['frac_tipped']*100:5.1f}% "
            f"early={stats['frac_stopped_early']*100:5.1f}% | "
            f"{finish_str} | "
            f"gen_time={stats['elapsed_s']:.1f}s",
            flush=True,
        )

print(f"training done in {time.perf_counter() - train_start:.1f}s over {len(history)} generations")

In [ ]:
import matplotlib.pyplot as plt

if not history:
    print("no training history yet; run the training cell first.")
else:
    gens = [h["generation"] for h in history]
    mean_r = [h["mean_reward"] for h in history]
    max_r = [h["max_reward"] for h in history]
    min_r = [h["min_reward"] for h in history]
    frac_reached = [h["frac_reached"] * 100.0 for h in history]
    frac_tipped = [h["frac_tipped"] * 100.0 for h in history]
    frac_early = [h["frac_stopped_early"] * 100.0 for h in history]
    finish_gens = [h["generation"] for h in history if h["finishers"] > 0]
    finish_times = [h["avg_finish_time_s"] for h in history if h["finishers"] > 0]

    fig, axes = plt.subplots(2, 2, figsize=(11, 7))

    ax = axes[0, 0]
    ax.fill_between(gens, min_r, max_r, alpha=0.2, label="min\u2013max range")
    ax.plot(gens, mean_r, label="mean reward", linewidth=2)
    ax.plot(gens, max_r, label="max reward", linestyle="--", linewidth=1)
    ax.set_xlabel("generation")
    ax.set_ylabel("episode reward")
    ax.set_title("reward per generation")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

    ax = axes[0, 1]
    ax.plot(gens, frac_reached, label="reached goal", color="tab:green", linewidth=2)
    ax.plot(gens, frac_tipped, label="tipped over", color="tab:red", linewidth=2)
    ax.plot(gens, frac_early, label="stopped early (any)", color="tab:gray", linestyle="--")
    ax.set_xlabel("generation")
    ax.set_ylabel("% of population rollouts")
    ax.set_ylim(0, 100)
    ax.set_title("early-termination rates")
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")

    ax = axes[1, 0]
    if finish_gens:
        ax.plot(finish_gens, finish_times, marker="o", color="tab:blue")
        ax.set_ylabel("avg finish time (s)")
    else:
        ax.text(0.5, 0.5, "no finishers yet", ha="center", va="center", transform=ax.transAxes)
    ax.set_xlabel("generation")
    ax.set_xlim(min(gens), max(gens))
    ax.set_title("avg time to reach goal (finishers only)")
    ax.grid(True, alpha=0.3)

    ax = axes[1, 1]
    gen_elapsed = [h["elapsed_s"] for h in history]
    ax.plot(gens, gen_elapsed, color="tab:purple")
    ax.set_xlabel("generation")
    ax.set_ylabel("wall-clock seconds")
    ax.set_title("generation duration")
    ax.grid(True, alpha=0.3)

    fig.tight_layout()
    plt.show()

In [ ]:
eval_reward, eval_result = evaluate(
    params,
    max_steps=MAX_EPISODE_STEPS,
    episodes=1,
    record_frames=True,
)
print(
    f"eval | reward={eval_reward:+.3f} steps={eval_result.steps} "
    f"reached={eval_result.reached_goal} tipped={eval_result.tipped} "
    f"init_dist={eval_result.initial_distance_m:.2f}m "
    f"final_dist={eval_result.final_distance_m:.2f}m "
    f"time={eval_result.steps * BRAIN_DT_S:.2f}s"
)

counts = np.asarray(eval_result.action_counts)
top_k = 8
top_idx = np.argsort(counts)[::-1][:top_k]
print("\ntop actions chosen during eval (action_index | per-leg deltas | count):")
from brains.harnesses import ACTION_DELTAS
for idx in top_idx:
    if counts[idx] == 0:
        break
    deltas = ACTION_DELTAS[idx].astype(int).tolist()
    print(f"  {int(idx):3d} | {deltas} | {int(counts[idx])}")

In [ ]:
# Save in the discoverable checkpoint layout: checkpoints/<run_id>/latest.npz + manifest.json.
# The live viewer's playback factory dispatches on architecture=discrete_leg_linear.
from dataclasses import replace
from brains.config import load_runtime_spec
from brains.runtime.model_store import create_model_run_paths, write_model_manifest

MODEL_TYPE = "discrete_leg_linear"
log_id = datetime.now(timezone.utc).strftime("discrete_leg_%Y%m%dT%H%M%SZ")
checkpoint_root = REPO_ROOT / "checkpoints"
paths = create_model_run_paths(checkpoint_root, MODEL_TYPE, log_id)
paths.run_dir.mkdir(parents=True, exist_ok=True)

np.savez(
    paths.latest_path,
    W=np.asarray(params["W"]),
    b=np.asarray(params["b"]),
    base_obs_dim=np.int32(harness.BASE_OBS_DIM),
    encoded_obs_dim=np.int32(ENCODED_DIM),
    num_actions=np.int32(NUM_ACTIONS),
    brain_dt_s=np.float32(BRAIN_DT_S),
    positional_encoding_gain=np.float32(POSITIONAL_ENCODING_GAIN),
    goal_reached_radius_m=np.float32(GOAL_REACHED_RADIUS_M),
    tip_kill_depth=np.float32(TIP_KILL_DEPTH),
    tip_kill_steps=np.int32(TIP_KILL_STEPS),
    goal_reached_bonus=np.float32(GOAL_REACHED_BONUS),
    tipped_penalty=np.float32(TIPPED_PENALTY),
    model_type=MODEL_TYPE,
    model_id=paths.id,
    log_id=paths.log_id,
    config_name="discrete-leg",
    generation=np.int32(len(history)),
    best_reward=np.float32(max((h["max_reward"] for h in history), default=0.0)),
    mean_reward=np.float32(history[-1]["mean_reward"] if history else 0.0),
)

# Build a RuntimeSpec that records architecture=discrete_leg_linear so the viewer dispatches correctly.
base_spec = load_runtime_spec(REPO_ROOT / "configs/default.yaml")
spec_for_manifest = replace(
    base_spec,
    name="discrete-leg",
    model=replace(
        base_spec.model,
        type=MODEL_TYPE,
        architecture="discrete_leg_linear",
        trainer="notebook_es",
        description="81-action discrete-leg linear policy trained in notebook.",
    ),
    episode=replace(base_spec.episode, brain_dt_s=float(BRAIN_DT_S)),
)
spec_for_manifest.validate()

manifest_path = write_model_manifest(
    paths,
    spec_for_manifest,
    paths.latest_path,
    generation=len(history),
    best_reward=float(max((h["max_reward"] for h in history), default=0.0)),
    mean_reward=float(history[-1]["mean_reward"] if history else 0.0),
)
print("checkpoint:", paths.latest_path)
print("manifest:  ", manifest_path)
print("model id:  ", paths.id)